In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from pathlib import Path

from config import (
    TAGS_EXCEL_PATH, DATA_CSV_PART1, DATA_CSV_PART2, TARGET_COL,
    FINAL_WEIGHTS, load_tag_lists,
)

# Целочисленные веса + strict equal-count группы: auto_lover

L1 LogReg -> целочисленные веса -> строго равные группы

In [ ]:
tags_descriptions = pd.read_excel(TAGS_EXCEL_PATH, sheet_name='HT_list')
tag_lists = load_tag_lists(tags_descriptions)
auto_lover_list = tag_lists['auto_lover_list']

part1 = pd.read_csv(DATA_CSV_PART1, encoding='cp1251', delimiter=',')
part2 = pd.read_csv(DATA_CSV_PART2, encoding='cp1251', delimiter=',')
casco_hashtags_full = pd.merge(part1, part2, on='POLICY_ZV', how='inner')
casco_hashtags_full[TARGET_COL] = casco_hashtags_full['CLAIMS_PART_DAM_COUNT'].astype(bool).astype(int)
casco_hashtags_full.set_index('POLICY_ZV', inplace=True)

casco_hashtags_full.drop(columns=['TAG_JOIN_IND'], inplace=True)
casco_hashtags_full['SUM'] = casco_hashtags_full.filter(like='TAG_').fillna(0).sum(axis=1)
casco_hashtags_full = casco_hashtags_full[casco_hashtags_full['SUM'] > 0]
print('shape:', casco_hashtags_full.shape)

## IntegerMonotonicLinearGrouper + grid search

In [ ]:
def _detect_monotonic_direction(rates, tol=1e-12):
    rates = np.asarray(rates, dtype=float)
    if rates.size <= 1:
        return 'flat', True, True
    diffs = np.diff(rates)
    non_decreasing = bool(np.all(diffs >= -tol))
    non_increasing = bool(np.all(diffs <= tol))
    if non_decreasing and non_increasing:
        direction = 'flat'
    elif non_decreasing:
        direction = 'up'
    elif non_increasing:
        direction = 'down'
    else:
        direction = None
    return direction, non_decreasing, non_increasing


In [ ]:
class IntegerMonotonicLinearGrouper:

    """
    Линейный скор по TAG с целочисленными весами + strict equal-count binning.

    grouping_strategy:
      - 'strict_equal_count' (рекомендуется для WTW/Emblem):
          группы строятся одинакового размера (если requested_groups не делит N,
          подбирается ближайший делитель N).
      - 'quantile': обычный quantile-binning (equal-frequency, но возможны перекосы из-за ties).

    use_pav:
      - False: сохраняем equal-count (для strict стратегии)
      - True: включаем PAV-схлопывание (монотонность, но группы перестают быть равными)
    """

    def __init__(
        self,
        n_groups=30,
        C=0.35,
        max_int_weight=10,
        min_nonzero_weight=1,
        random_state=42,
        use_pav=False,
        grouping_strategy='strict_equal_count',
        enforce_exact_equal=True
    ):
        self.n_groups = int(n_groups)
        self.C = float(C)
        self.max_int_weight = int(max_int_weight)
        self.min_nonzero_weight = int(min_nonzero_weight)
        self.random_state = random_state
        self.use_pav = bool(use_pav)
        self.grouping_strategy = str(grouping_strategy)
        self.enforce_exact_equal = bool(enforce_exact_equal)

        self.columns_ = None
        self.weights_raw_ = None
        self.weights_ = None
        self.prebin_edges_ = None
        self.prebin_to_group_ = None
        self.prebin_stats_ = None
        self.group_stats_ = None
        self.n_groups_actual_ = None
        self.n_groups_requested_ = int(n_groups)
        self.n_groups_effective_ = int(n_groups)
        self.equal_group_size_ = None

    def _build_integer_weights(self, raw_weights):
        w = raw_weights.copy().clip(lower=0)
        if (w > 0).sum() == 0:
            w = raw_weights.abs()
        if (w > 0).sum() == 0:
            w = pd.Series(1.0, index=raw_weights.index)

        w_norm = w / w.max()
        w_int = (w_norm * self.max_int_weight).round().astype(int)
        mask_nonzero = w > 0
        w_int.loc[mask_nonzero & (w_int == 0)] = self.min_nonzero_weight
        w_int = w_int.clip(lower=0, upper=self.max_int_weight)
        return w_int

    def _nearest_divisor(self, n, target):
        # Ищем делитель n (>=3), ближайший к target, но не выше target.
        divisors = [g for g in range(3, n + 1) if n % g == 0]
        if not divisors:
            return None

        not_above = [g for g in divisors if g <= target]
        if not_above:
            # Самый близкий к target снизу.
            return max(not_above)

        # Если ниже target делителей нет, берем минимально возможный.
        return min(divisors)

    def _assign_groups_strict(self, score, n_groups):
        s = pd.Series(score).astype(float)
        n = len(s)
        g = int(max(3, min(int(n_groups), n)))

        # Детерминированный порядок: score, затем индекс
        idx_as_str = s.index.astype(str).to_numpy()
        order = np.lexsort((idx_as_str, s.to_numpy()))

        labels = np.empty(n, dtype=int)
        per_group = n // g
        start = 0
        for k in range(g):
            end = start + per_group
            labels[order[start:end]] = k
            start = end

        return pd.Series(labels, index=s.index, dtype=int)

    def fit(self, df, cols, y, feature_name='feature'):
        if self.n_groups < 3:
            raise ValueError('n_groups должен быть >= 3')
        if self.max_int_weight < 1:
            raise ValueError('max_int_weight должен быть >= 1')
        if self.grouping_strategy not in ('strict_equal_count', 'quantile'):
            raise ValueError("grouping_strategy должен быть 'strict_equal_count' или 'quantile'")
        if self.grouping_strategy == 'strict_equal_count' and self.use_pav:
            raise ValueError("Для strict_equal_count use_pav=True противоречит цели равных групп")

        selected_cols = [c for c in cols if c in df.columns]
        if not selected_cols:
            raise ValueError(f"Не найдено ни одного TAG для '{feature_name}' в данных.")

        X_full = df[selected_cols].fillna(0).astype(float)
        y = pd.Series(y, index=df.index).astype(int)

        non_const = X_full.nunique(dropna=False)
        model_cols = non_const[non_const > 1].index.tolist()
        if not model_cols:
            raise ValueError(f"Все TAG для '{feature_name}' оказались константными.")

        X_model = X_full[model_cols]
        logreg = LogisticRegression(
            penalty='l1',
            solver='saga',
            C=self.C,
            max_iter=6000,
            random_state=self.random_state,
            n_jobs=-1
        )
        logreg.fit(X_model, y)

        self.weights_raw_ = pd.Series(logreg.coef_.ravel(), index=model_cols)
        weights_int_all = self._build_integer_weights(self.weights_raw_)
        weights_int_all = weights_int_all[weights_int_all > 0]
        if weights_int_all.empty:
            weights_int_all = pd.Series(1, index=model_cols, dtype=int)

        self.columns_ = weights_int_all.index.tolist()
        self.weights_ = weights_int_all.astype(int)

        X = X_full[self.columns_]
        score = X.dot(self.weights_)

        n_obs = len(score)
        self.n_groups_requested_ = int(self.n_groups)
        self.n_groups_effective_ = int(self.n_groups)

        if self.grouping_strategy == 'strict_equal_count':
            if self.enforce_exact_equal and (n_obs % self.n_groups_effective_ != 0):
                adjusted = self._nearest_divisor(n_obs, self.n_groups_effective_)
                if adjusted is None:
                    raise ValueError('Не удалось подобрать делитель для exact equal групп')
                self.n_groups_effective_ = int(adjusted)

            prebin_series = self._assign_groups_strict(score, self.n_groups_effective_)
            prebin = prebin_series.values
            self.prebin_edges_ = None
            self.prebin_to_group_ = {int(g): int(g) for g in sorted(prebin_series.unique())}
        else:
            quantiles = np.linspace(0, 1, self.n_groups_effective_ + 1)
            edges = np.quantile(score, quantiles)
            edges = np.unique(edges)
            if len(edges) < 2:
                edges = np.array([score.min(), score.max() + 1e-9])
            self.prebin_edges_ = edges
            prebin = np.digitize(score, self.prebin_edges_[1:-1], right=True)
            self.prebin_to_group_ = {int(g): int(g) for g in sorted(np.unique(prebin))}

        self.prebin_stats_ = (
            pd.DataFrame({'prebin': prebin, 'target': y.values})
            .groupby('prebin', as_index=True)['target']
            .agg(['mean', 'count'])
            .sort_index()
        )

        groups = pd.Series(prebin, index=df.index).map(self.prebin_to_group_).astype(int)
        self.group_stats_ = (
            pd.DataFrame({'group': groups, 'target': y.values})
            .groupby('group', as_index=True)['target']
            .agg(['mean', 'count'])
            .sort_index()
            .rename(columns={'mean': 'claim_rate', 'count': 'objects'})
        )
        self.n_groups_actual_ = int(self.group_stats_.shape[0])
        self.equal_group_size_ = bool(self.group_stats_['objects'].nunique() == 1)
        return self

    def transform(self, df):
        if self.columns_ is None or self.weights_ is None:
            raise ValueError('Сначала вызовите fit().')

        X = df.reindex(columns=self.columns_, fill_value=0).fillna(0).astype(float)
        score = X.dot(self.weights_)

        if self.grouping_strategy == 'strict_equal_count':
            prebin = self._assign_groups_strict(score, self.n_groups_effective_).values
        else:
            prebin = np.digitize(score, self.prebin_edges_[1:-1], right=True)

        groups = pd.Series(prebin, index=df.index).map(self.prebin_to_group_).astype(int)
        return groups, score


def fit_monotonic_integer_candidates(
    df,
    feature_cols,
    y,
    candidate_groups=tuple(range(3, 31)),
    C=0.35,
    max_int_weight=10,
    min_nonzero_weight=1,
    random_state=42,
    feature_name='feature',
    verbose=True,
    use_pav=False,
    grouping_strategy='strict_equal_count',
    enforce_exact_equal=True
):
    candidate_groupers = {}
    summary_rows = []

    for n in candidate_groups:
        g = IntegerMonotonicLinearGrouper(
            n_groups=n,
            C=C,
            max_int_weight=max_int_weight,
            min_nonzero_weight=min_nonzero_weight,
            random_state=random_state,
            use_pav=use_pav,
            grouping_strategy=grouping_strategy,
            enforce_exact_equal=enforce_exact_equal
        )
        g.fit(df, feature_cols, y, feature_name=feature_name)
        candidate_groupers[int(n)] = g

        monotonic_ok = bool(np.all(np.diff(g.group_stats_['claim_rate'].values) >= -1e-12))
        summary_rows.append({
            'feature': feature_name,
            'requested_groups': int(n),
            'effective_groups': int(g.n_groups_effective_),
            'actual_groups': int(g.n_groups_actual_),
            'equal_group_size': bool(g.equal_group_size_),
            'monotonic_non_decreasing': monotonic_ok,
            'n_tags_with_weight': int((g.weights_ > 0).sum()),
            'max_integer_weight': int(g.weights_.max()),
            'grouping_strategy': grouping_strategy,
            'enforce_exact_equal': bool(enforce_exact_equal)
        })

        if verbose:
            print()
            print(
                f"=== {feature_name}: requested={n}, effective={g.n_groups_effective_}, actual={g.n_groups_actual_}, "
                f"equal_group_size={g.equal_group_size_}, monotonic={monotonic_ok}, tags={(g.weights_ > 0).sum()} ==="
            )
            print(g.group_stats_)

    summary_df = pd.DataFrame(summary_rows).sort_values('requested_groups').reset_index(drop=True)
    return candidate_groupers, summary_df


def apply_monotonic_integer_feature(
    df,
    out_df,
    output_col,
    feature_cols,
    y,
    selected_groups=30,
    candidate_groups=tuple(range(3, 31)),
    C=0.35,
    max_int_weight=10,
    min_nonzero_weight=1,
    random_state=42,
    feature_name='feature',
    verbose=True,
    use_pav=False,
    grouping_strategy='strict_equal_count',
    enforce_exact_equal=True
):
    candidate_groupers, summary_df = fit_monotonic_integer_candidates(
        df=df,
        feature_cols=feature_cols,
        y=y,
        candidate_groups=candidate_groups,
        C=C,
        max_int_weight=max_int_weight,
        min_nonzero_weight=min_nonzero_weight,
        random_state=random_state,
        feature_name=feature_name,
        verbose=verbose,
        use_pav=use_pav,
        grouping_strategy=grouping_strategy,
        enforce_exact_equal=enforce_exact_equal
    )

    if int(selected_groups) in candidate_groupers:
        selected_grouper = candidate_groupers[int(selected_groups)]
    else:
        selected_grouper = IntegerMonotonicLinearGrouper(
            n_groups=selected_groups,
            C=C,
            max_int_weight=max_int_weight,
            min_nonzero_weight=min_nonzero_weight,
            random_state=random_state,
            use_pav=use_pav,
            grouping_strategy=grouping_strategy,
            enforce_exact_equal=enforce_exact_equal
        )
        selected_grouper.fit(df, feature_cols, y, feature_name=feature_name)

    groups, score = selected_grouper.transform(df)
    out_df[output_col] = groups
    return selected_grouper, candidate_groupers, summary_df, score


def rank_tags_for_integer_model(df, feature_cols, y, C=0.35, max_int_weight=10, min_nonzero_weight=1, random_state=42):
    existing_cols = [c for c in feature_cols if c in df.columns]
    if not existing_cols:
        raise ValueError('Нет доступных TAG для ранжирования')

    # Используем технический фит для получения ранга TAG по integer weight.
    g_rank = IntegerMonotonicLinearGrouper(
        n_groups=10,
        C=C,
        max_int_weight=max_int_weight,
        min_nonzero_weight=min_nonzero_weight,
        random_state=random_state,
        use_pav=False,
        grouping_strategy='strict_equal_count',
        enforce_exact_equal=False
    )
    g_rank.fit(df, existing_cols, y, feature_name='rank_integer')
    ranked_weights = g_rank.weights_.sort_values(ascending=False)
    ranked_tags = ranked_weights.index.tolist()
    return ranked_tags, ranked_weights


def search_integer_equalcount_dynamic(
    df,
    feature_cols,
    y,
    candidate_groups=tuple(range(3, 31)),
    candidate_tag_counts=(8, 12, 16, 20, None),
    trend_mode='both',  # 'both', 'up', 'down', 'up_soft'
    soft_min_up_share=0.70,
    soft_min_slope=0.0,
    C=0.35,
    max_int_weight=10,
    min_nonzero_weight=1,
    random_state=42,
    verbose=True
):
    if trend_mode not in ('both', 'up', 'down', 'up_soft'):
        raise ValueError("trend_mode должен быть 'both'|'up'|'down'|'up_soft'")

    ranked_tags, ranked_weights = rank_tags_for_integer_model(
        df=df,
        feature_cols=feature_cols,
        y=y,
        C=C,
        max_int_weight=max_int_weight,
        min_nonzero_weight=min_nonzero_weight,
        random_state=random_state
    )

    if not ranked_tags:
        raise ValueError('После ранжирования не осталось TAG')

    tag_counts = []
    for k in candidate_tag_counts:
        if k is None:
            tag_counts.append(len(ranked_tags))
        else:
            tag_counts.append(int(k))
    tag_counts = sorted({k for k in tag_counts if 1 <= k <= len(ranked_tags)})
    if len(ranked_tags) not in tag_counts:
        tag_counts.append(len(ranked_tags))

    rows = []
    models = {}

    for k in tag_counts:
        subset = ranked_tags[:k]
        for n in candidate_groups:
            g = IntegerMonotonicLinearGrouper(
                n_groups=int(n),
                C=C,
                max_int_weight=max_int_weight,
                min_nonzero_weight=min_nonzero_weight,
                random_state=random_state,
                use_pav=False,
                grouping_strategy='strict_equal_count',
                enforce_exact_equal=True
            )
            g.fit(df, subset, y, feature_name=f'integer_k{k}_g{n}')

            claim_rates = g.group_stats_['claim_rate'].values.astype(float)
            direction, mono_up, mono_down = _detect_monotonic_direction(claim_rates)
            dynamic_range = float(claim_rates.max() - claim_rates.min())
            diffs = np.diff(claim_rates)
            up_share = float((diffs >= -1e-12).mean()) if len(diffs) > 0 else 1.0
            slope = float(np.polyfit(np.arange(len(claim_rates)), claim_rates, 1)[0]) if len(claim_rates) > 1 else 0.0
            total_drop = float((-diffs[diffs < 0]).sum()) if len(diffs) > 0 else 0.0

            if trend_mode == 'both':
                trend_ok = direction in ('up', 'down')
            elif trend_mode == 'up':
                trend_ok = direction == 'up'
            elif trend_mode == 'down':
                trend_ok = direction == 'down'
            else:  # up_soft
                trend_ok = bool((up_share >= float(soft_min_up_share)) and (slope > float(soft_min_slope)))

            rows.append({
                'tag_count': int(k),
                'requested_groups': int(n),
                'effective_groups': int(g.n_groups_effective_),
                'actual_groups': int(g.n_groups_actual_),
                'group_distance': int(abs(int(g.n_groups_effective_) - int(n))),
                'equal_group_size': bool(g.equal_group_size_),
                'trend_direction': direction,
                'monotonic_non_decreasing': bool(mono_up),
                'monotonic_non_increasing': bool(mono_down),
                'up_share': float(up_share),
                'slope': float(slope),
                'total_drop': float(total_drop),
                'trend_ok': bool(trend_ok),
                'claim_rate_range': float(dynamic_range)
            })

            models[(int(k), int(n))] = (g, subset)

            if verbose:
                print(
                    f"k={k:>3}, req={n:>3}, eff={g.n_groups_effective_:>3}, "
                    f"trend={direction}, range={dynamic_range:.6f}, equal={g.equal_group_size_}"
                )

    search_df = pd.DataFrame(rows).sort_values(
        ['trend_ok', 'up_share', 'slope', 'claim_rate_range', 'total_drop', 'group_distance', 'effective_groups', 'tag_count'],
        ascending=[False, False, False, False, True, True, False, True]
    ).reset_index(drop=True)

    if search_df.empty:
        return None, None, search_df, ranked_weights

    # Предпочитаем trend_ok=True, но если нет — берем лучший компромисс по up_share/slope.
    search_df_trend = search_df[search_df['trend_ok']].reset_index(drop=True)
    if search_df_trend.empty:
        print(
            f"WARNING: для '{trend_mode}' не найдено строгого совпадения; выбран лучший компромисс (up_share/slope)."
        )
        best = search_df.iloc[0]
    else:
        best = search_df_trend.iloc[0]
    best_key = (int(best['tag_count']), int(best['requested_groups']))
    best_model, best_subset = models[best_key]

    return best_model, best_subset, search_df, ranked_weights

## Поиск оптимальной конфигурации для auto_lover

In [ ]:
# --- Пример: поиск комбинации TAG + groups с динамикой (strict equal-count) для auto_lover ---

new_features_monotonic_integer = pd.DataFrame(index=casco_hashtags_full.index)

y_auto = casco_hashtags_full['BINARY_CLAIMS_PART_DAM_COUNT']

best_grouper_int, best_tags_int, search_df_int, ranked_weights_int = search_integer_equalcount_dynamic(
    df=casco_hashtags_full,
    feature_cols=auto_lover_list,
    y=y_auto,
    candidate_groups=list(range(20, 31)),
    candidate_tag_counts=[8, 12, 16, 20, None],
    trend_mode='up_soft',  # мягко восходящий тренд
    C=0.35,
    max_int_weight=10,
    min_nonzero_weight=1,
    random_state=42,
    soft_min_up_share=0.70,
    soft_min_slope=0.0,
    verbose=False
)

if best_grouper_int is None:
    raise ValueError('Не удалось подобрать ни одной конфигурации')

groups_int, auto_score_int = best_grouper_int.transform(casco_hashtags_full)
new_features_monotonic_integer['agg_auto_equalcount_integer'] = groups_int

print('Top-20 TAG по integer-весу (ранжирование):')
print(ranked_weights_int.head(20).to_string())

print()
print('Поиск конфигураций (top-20):')
print(search_df_int.head(20).to_string(index=False))

print()
print('Выбранная конфигурация:')
print(f"tag_count={len(best_tags_int)}")
print(f"requested_groups={best_grouper_int.n_groups_requested_}")
print(f"effective_groups={best_grouper_int.n_groups_effective_}")
print(f"actual_groups={best_grouper_int.n_groups_actual_}")
print(f"equal_group_size={best_grouper_int.equal_group_size_}")

check_df_monotonic_int = pd.concat([
    new_features_monotonic_integer[['agg_auto_equalcount_integer']],
    y_auto
], axis=1)

analysis_monotonic_int = (
    check_df_monotonic_int.groupby('agg_auto_equalcount_integer', as_index=True)['BINARY_CLAIMS_PART_DAM_COUNT']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'claim_rate', 'count': 'objects'})
    .sort_index()
)

print()
print('Анализ agg_auto_equalcount_integer:')
print(analysis_monotonic_int)

objects_min = int(analysis_monotonic_int['objects'].min())
objects_max = int(analysis_monotonic_int['objects'].max())
objects_ratio = float(objects_max / objects_min) if objects_min > 0 else np.nan
print()
print(f"Размеры групп (min/max): {objects_min}/{objects_max}, ratio={objects_ratio:.3f}")
print(f"Строго одинаковые группы: {objects_min == objects_max}")

direction_int, mono_up_int, mono_down_int = _detect_monotonic_direction(analysis_monotonic_int['claim_rate'].values)
print(f"Направление тренда: {direction_int}")
print(f"Неубывающий={mono_up_int}, невозрастающий={mono_down_int}")

plt.figure(figsize=(11, 5))
plt.bar(analysis_monotonic_int.index.astype(str), analysis_monotonic_int['claim_rate'], color='steelblue')
plt.title('agg_auto_equalcount_integer: частота аварий')
plt.ylabel('Средняя частота аварий')
plt.xlabel('Equal-count группа')
plt.tight_layout()
plt.show()

# Расшифровка TAG -> имя + integer weight (только выбранные TAG)
used_int = (
    best_grouper_int.weights_
    .rename('integer_weight')
    .reset_index()
    .rename(columns={'index': 'TAG'})
    .sort_values('integer_weight', ascending=False)
)
tag_map = tags_descriptions[['TAG', 'TAG_NAME fin SBS']].drop_duplicates()
used_int = used_int.merge(tag_map, on='TAG', how='left')

print()
print('Топ-30 TAG с целыми весами (выбранная конфигурация):')
print(used_int[['TAG', 'TAG_NAME fin SBS', 'integer_weight']].head(30).to_string(index=False))


In [ ]:
print()
GROUP_KEY = 'auto_lover'
weights_for_export = used_int[['TAG', 'integer_weight']].copy()
weights_for_export['integer_weight'] = weights_for_export['integer_weight'].astype(int)
weights_dict = dict(zip(weights_for_export['TAG'], weights_for_export['integer_weight']))

print(f"'{GROUP_KEY}': {{")
for tag, integer_weight in weights_dict.items():
    print(f"    '{tag}': {int(integer_weight)},")
print('}')

In [ ]:
ARTIFACTS_DIR = Path('artifacts')
CSV_DIR = ARTIFACTS_DIR / 'csv'
for p in (CSV_DIR,):
    p.mkdir(parents=True, exist_ok=True)

# Сохранение результатов
if best_grouper_int is not None:
    groups_int, _ = best_grouper_int.transform(casco_hashtags_full)
    result_df = pd.DataFrame({
        'agg_auto_equalcount_integer': groups_int
    }, index=casco_hashtags_full.index)
    result_df.to_csv(CSV_DIR / 'agg_auto_equalcount_integer.csv', encoding='utf-8-sig')
    print(f'Сохранено: {CSV_DIR / "agg_auto_equalcount_integer.csv"}')